In [2]:
import sys; sys.path.append("..")      # para encontrar src/
import time
from multiprocessing import Pool, cpu_count
from pymongo import MongoClient
from pymongo.errors import BulkWriteError
from tqdm.auto import tqdm              # tqdm.auto se ve bien en notebooks

from src.kmers import procesar_lote

db = MongoClient("mongodb://localhost:27017")["panteon_prueba"]
db.kmers.create_index([("secuencia_id", 1), ("k", 1)], unique=True)
db.kmers.create_index([("te_orden", 1), ("k", 1)])

'te_orden_1_k_1'

In [3]:
ya = {d["secuencia_id"] for d in db.kmers.find({"k": 6}, {"secuencia_id": 1})}
total = db.secuencias.count_documents({})
print(f"total={total}  ya hechas={len(ya)}")

def lotes(tam=200, maximo=None):
    cursor = db.secuencias.find({}, {"te_orden": 1, "secuencia": 1})
    if maximo:
        cursor = cursor.limit(maximo)
    lote = []
    for s in cursor:
        if s["_id"] in ya:
            continue
        lote.append((s["_id"], s["te_orden"], s["secuencia"]))
        if len(lote) == tam:
            yield lote
            lote = []
    if lote:
        yield lote

total=239946  ya hechas=239946


In [4]:
def insertar(docs):
    try:
        db.kmers.insert_many(docs, ordered=False)
    except BulkWriteError as e:
        if any(err["code"] != 11000 for err in e.details["writeErrors"]):
            raise

t0 = time.perf_counter()
with Pool(max(1, cpu_count() - 1)) as pool, tqdm(total=total, initial=len(ya)) as barra:
    for docs in pool.imap(procesar_lote, lotes(maximo=1000)):   # quita maximo para todo
        insertar(docs)
        barra.update(len(docs) // 6)
print(f"{time.perf_counter() - t0:.1f} s")

100%|██████████| 239946/239946 [00:00<?, ?it/s]

2.8 s


In [5]:
# 1. Totales
print(db.kmers.count_documents({}), 6 * db.secuencias.count_documents({}))

# 2. Secuencias con ventanas descartadas (N u otros caracteres)
pipeline = [
    {"$match": {"k": 1}},
    {"$lookup": {"from": "secuencias", "localField": "secuencia_id",
                 "foreignField": "_id", "as": "s"}},
    {"$unwind": "$s"},
    {"$match": {"$expr": {"$lt": ["$total_ventanas", "$s.longitud"]}}},
    {"$count": "con_descartes"},
]
print(list(db.kmers.aggregate(pipeline)))

1439676 1439676
[{'con_descartes': 54837}]


In [7]:
print("Documento en la base de datos:")
kmer = db.kmers.find_one({"k": 6})
print(f"id secuencia: {kmer['secuencia_id']}")
print(f"te_orden: {kmer['te_orden']}")
print(f"total_ventanas: {kmer['total_ventanas']}")
print(f"conteos: {kmer['conteos']}")

Documento en la base de datos:
id secuencia: 6ae27e4ffa276374128bf63b
te_orden: HELITRON
total_ventanas: 1986
conteos: {'AAAAAA': 18, 'AAAAAC': 1, 'AAAACA': 2, 'AAACAA': 2, 'AACAAA': 2, 'ACAAAA': 2, 'CAAAAG': 3, 'AAAAGA': 2, 'AAAGAA': 4, 'AAGAAC': 1, 'AGAACT': 2, 'GAACTT': 1, 'AACTTA': 1, 'ACTTAA': 1, 'CTTAAA': 2, 'TTAAAT': 5, 'TAAATT': 4, 'AAATTT': 13, 'AATTTT': 9, 'ATTTTT': 8, 'TTTTTT': 5, 'TTTTTA': 6, 'TTTTAA': 14, 'TTTAAA': 15, 'TTAAAA': 13, 'TAAAAT': 6, 'AAAATT': 11, 'AATTTC': 6, 'ATTTCA': 4, 'TTTCAA': 6, 'TTCAAT': 2, 'TCAATA': 1, 'CAATAT': 3, 'AATATA': 4, 'ATATAA': 2, 'TATAAA': 3, 'ATAAAA': 5, 'TAAAAA': 11, 'AAAAAT': 14, 'ATTTTA': 8, 'TAAATC': 2, 'AAATCA': 5, 'AATCAA': 2, 'ATCAAA': 3, 'TCAAAA': 6, 'CAAAAA': 4, 'AAATTA': 4, 'AATTAT': 2, 'ATTATT': 5, 'TTATTA': 5, 'TATTAA': 6, 'ATTAAA': 6, 'TTCAAA': 5, 'AAAATA': 6, 'AAATAA': 6, 'AATAAA': 6, 'TCAAAT': 3, 'CAAATT': 2, 'AATTTA': 2, 'ATTTAA': 3, 'AAAATC': 3, 'AATCAT': 4, 'ATCATA': 3, 'TCATAT': 2, 'CATATA': 1, 'ATATAT': 8, 'TATATT': 4, '